# Fermi-Dirac equilibrium test

This notebook checks that the neutrino occupation numbers in the Fermi-Dirac simulation match the analytic Fermi-Dirac distribution. Electron, muon, and tau neutrinos and antineutrinos are each required to agree to a maximum relative error below 0.01.

Converted from `/home/erick/software/Emu/Scripts/tests/fermi_dirac_test.py`. Physics, numerical values, tolerances, and checks are unchanged.

### Imports and paths

Load the libraries used by the test. `EMU_DIR` is the Emu source tree. `DATA_DIR` is the Fermi-Dirac run that contains the `plt*.h5` plotfiles. The original script globbed those files from the working directory; the notebook reads them from `DATA_DIR` so it can be run from `notebooks/`.

In [ ]:
%matplotlib inline

import glob
import os
from pathlib import Path
import sys
import h5py
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import AutoLocator, AutoMinorLocator, LogLocator

try:
    here = Path(__file__).resolve().parent
except NameError:
    here = Path.cwd()

sys.path.append(str((here / '..' / 'plot_functions').resolve()))
sys.path.append(str((here / '..' / 'phys_const').resolve()))

import plot_functions as pf
import phys_const as pc
from phys_const import CGSUnitsConst

EMU_DIR = "/home/erick/software/Emu"
DATA_DIR = (
    "/mnt/scratch/erick/erick_scratch/emu_methods_paper_simulations/"
    "cassowary/fermi-dirac"
)

print(f"EMU_DIR = {EMU_DIR}")
print(f"DATA_DIR = {DATA_DIR}")

### Parameters and constants

Physical constants come from `phys_const`. NuLib energy-bin edges and centers, background temperature, and chemical potentials match the Fermi-Dirac test script.

In [ ]:
#
# Energy bins from NuLib table (Cutted)
# The following energy bins should be the same as the st9_empty_particles_multi_energy initial condition script
#
# Energy bin centers extracted from NuLib table
energies_center_Mev = np.array([15.6909, 20.9527, 27.4681, 35.5357, 45.5254, 57.8951, 73.2117, 92.1775, 115.662, 144.741, 180.748, 225.334, 280.542]) # Energy in Mev
# Energy bin bottom extracted from NuLib table
energies_bottom_Mev = np.array([13.3401, 18.0418, 23.8636, 31.0725, 39.9989, 51.0519, 64.7382, 81.6853, 102.67, 128.654, 160.828, 200.668, 250])
# Energy bin top extracted from NuLib table
energies_top_Mev = np.array([18.0418, 23.8636, 31.0725, 39.9989, 51.0519, 64.7382, 81.6853, 102.67, 128.654, 160.828, 200.668, 250, 311.085])

# Energies in ergs
energies_center_erg = np.array(energies_center_Mev) * 1e6 * CGSUnitsConst.eV # Energy in ergs
energies_bottom_erg = np.array(energies_bottom_Mev) * 1e6 * CGSUnitsConst.eV # Energy in ergs
energies_top_erg    = np.array(energies_top_Mev   ) * 1e6 * CGSUnitsConst.eV # Energy in ergs

# Simulation parameters
Temperature_Mev = 5.02464 # Background matter temperature ( Mev )
u_ee_MeV = -1.640496 # Electron neutrino chemical potential ( Mev )
u_eebar_MeV = +1.640496 # Electron anti-neutrino chemical potential ( Mev )
u_uu_MeV = 0.0 # Muon neutrino chemical potential ( Mev )
u_uubar_MeV = 0.0 # Muon anti-neutrino chemical potential ( Mev )
u_tt_MeV = 0.0 # Tauon neutrino chemical potential ( Mev )
u_ttbar_MeV = 0.0 # Tauon anti-neutrino chemical potential ( Mev )

# Absorption inverse mean free paths for electron neutrinos
k_ee_invcm = [9.127769017962591e-9, 1.641327312232395e-8, 2.8393746562301453e-8, 4.779390212302282e-8, 7.889848522592262e-8, 1.2841789305504384e-7, 2.0681101404663153e-7, 3.3030480560198965e-7, 5.239343668574352e-7, 8.26021165299387e-7, 0.0000012945991761253304, 0.000002016351227045427, 0.000003118591856951728, 0.000004784725631726817]
k_eebar_invcm = [4.6087508110584445e-9, 8.904111598164803e-9, 1.588826655111287e-8, 2.6804132945847618e-8, 4.327513760052642e-8, 6.732535548648654e-8, 1.0134311590252218e-7, 1.4794196172344386e-7, 2.0970630239629998e-7, 2.888967213808391e-7, 3.8735306743334104e-7, 5.070938475015653e-7, 6.5244112289689e-7, 8.348017555003779e-7]
k_xx_invcm = [1.0289145330897014e-9, 1.405712143835921e-9, 1.9125077764262436e-9, 2.5020221029371226e-9, 3.2551291182356493e-9, 4.302702684451385e-9, 5.385293260492072e-9, 6.977135451742849e-9, 8.697577392317867e-9, 1.1003787019861907e-8, 1.3807620922454043e-8, 1.727451168197901e-8, 2.156078723288311e-8, 2.6852329353452016e-8]

t_max_for_sim = np.log(0.01) /(-1.0 * np.min(k_ee_invcm)*pc.PhysConst.c)
t_min_for_sim = np.log(0.01) /(-1.0 * np.max(k_ee_invcm)*pc.PhysConst.c)
print(f't_max_for_sim = {t_max_for_sim}')
print(f't_min_for_sim = {t_min_for_sim}')

### Analytic Fermi-Dirac distribution

Equilibrium occupation number at each energy-bin center,

$$f_{\mathrm{eq}}(E) = \left[1 + \exp\left(\frac{E - \mu}{T}\right)\right]^{-1}.$$

$$f = (hc)^3 \, N \, / \, V_{\mathrm{phase}}.$$

In [ ]:
# Fermi-dirac distribution factor for electron neutrinos
f_eq_ee = 1 / ( 1 + np.exp( ( energies_center_Mev - u_ee_MeV ) / Temperature_Mev ) ) # adimentional
f_eq_eebar = 1 / ( 1 + np.exp( ( energies_center_Mev - u_eebar_MeV ) / Temperature_Mev ) ) # adimentional
f_eq_uu = 1 / ( 1 + np.exp( ( energies_center_Mev - u_uu_MeV ) / Temperature_Mev ) ) # adimentional
f_eq_uubar = 1 / ( 1 + np.exp( ( energies_center_Mev - u_uubar_MeV ) / Temperature_Mev ) ) # adimentional
f_eq_tt = 1 / ( 1 + np.exp( ( energies_center_Mev - u_tt_MeV ) / Temperature_Mev ) ) # adimentional
f_eq_ttbar = 1 / ( 1 + np.exp( ( energies_center_Mev - u_ttbar_MeV ) / Temperature_Mev ) ) # adimentional

### Plotfiles

Find every `plt*.h5` file in `DATA_DIR`, drop names that contain `old`, and sort by timestep number.

In [ ]:
# Reading plt* directories
directories = np.array(glob.glob(os.path.join(DATA_DIR, "plt*.h5")))
if directories.size == 0:
    raise FileNotFoundError(f"No plt*.h5 files found in {DATA_DIR}")
# Extract directories with "old" in their names
mask = np.char.find(directories, "old") == -1
directories = directories[mask]

# Sort the data file names by time step number
directories = sorted(directories, key=lambda x: int(x.split("plt")[1].split(".")[0]))

print(f"Found {len(directories)} plotfiles")
print("Last plotfile:", directories[-1])

### Equilibrium particle numbers

Phase-space volume of each energy bin and the Fermi-Dirac particle numbers drawn as the reference curves for electron and muon neutrinos and antineutrinos.

In [ ]:
Volume = 2.2e6**3 # ccm
Tot_Vphase = Volume * ( 4.0 * np.pi ) * ( ( energies_top_erg ** 3 - energies_bottom_erg ** 3 ) / 3.0 )

N_eq_ee = ( 1.0 / pc.PhysConst.hc**3 ) * Tot_Vphase * f_eq_ee
N_eq_uu = ( 1.0 / pc.PhysConst.hc**3 ) * Tot_Vphase * f_eq_uu
N_eq_eebar = ( 1.0 / pc.PhysConst.hc**3 ) * Tot_Vphase * f_eq_eebar
N_eq_uubar = ( 1.0 / pc.PhysConst.hc**3 ) * Tot_Vphase * f_eq_uubar

### Distribution functions and particle numbers

Occupation number and particle number versus energy at every snapshot, with the analytic Fermi-Dirac curve overlaid. Figures are shown inline.

In [ ]:
f_ab_all_times = np.zeros((len(directories),2,2,len(energies_center_erg)))
N_ab_all_times = np.zeros((len(directories),2,2,len(energies_center_erg)))
time_s = np.zeros(len(directories))
energies_center_MeV_cm = energies_center_erg / (CGSUnitsConst.eV * 1e6)


print(f_ab_all_times.shape)
print(N_ab_all_times.shape)

for i in range(len(directories)):
    with h5py.File(directories[i], 'r') as hf:

        N00_Re_this_dir = np.array(hf['N00_Re']) # number of particles
        N11_Re_this_dir = np.array(hf['N11_Re']) # number of particles
        N00_Rebar_this_dir = np.array(hf['N00_Rebar']) # number of particles
        N11_Rebar_this_dir = np.array(hf['N11_Rebar']) # number of particles
        E_this_dir = np.array(hf['pupt']) # ergs
        t_this_dir = np.array(hf['time']) # seconds
        Vphase_this_dir = np.array(hf['Vphase']) # seconds

        time_s[i] = t_this_dir[0]

        for j, energy_erg in enumerate(energies_center_erg):
            mask = E_this_dir == energy_erg
            N00_Re_this_dir_this_energy = np.sum(N00_Re_this_dir[mask])
            N00_Rebar_this_dir_this_energy = np.sum(N00_Rebar_this_dir[mask])
            N11_Re_this_dir_this_energy = np.sum(N11_Re_this_dir[mask])
            N11_Rebar_this_dir_this_energy = np.sum(N11_Rebar_this_dir[mask])
            Vphase_this_dir_this_energy = np.sum(Vphase_this_dir[mask])

            f_ab_all_times[i,0,0,j] = pc.PhysConst.hc**3 * N00_Re_this_dir_this_energy / Vphase_this_dir_this_energy
            f_ab_all_times[i,0,1,j] = pc.PhysConst.hc**3 * N00_Rebar_this_dir_this_energy / Vphase_this_dir_this_energy
            f_ab_all_times[i,1,0,j] = pc.PhysConst.hc**3 * N11_Re_this_dir_this_energy / Vphase_this_dir_this_energy
            f_ab_all_times[i,1,1,j] = pc.PhysConst.hc**3 * N11_Rebar_this_dir_this_energy / Vphase_this_dir_this_energy

            N_ab_all_times[i,0,0,j] = N00_Re_this_dir_this_energy
            N_ab_all_times[i,0,1,j] = N00_Rebar_this_dir_this_energy
            N_ab_all_times[i,1,0,j] = N11_Re_this_dir_this_energy
            N_ab_all_times[i,1,1,j] = N11_Rebar_this_dir_this_energy

pf.plot_colored_lines(
    np.log10(energies_center_MeV_cm),                         # x-axis: space/grid centers, shape (1024,)
    np.log10(f_ab_all_times[:, 0, 0, :]),                      # y values: need shape (space, time)
    time_s,                                 # t-axis: list of times
    r'$\log_{10}(E) \, [\mathrm{MeV}]$',                # x label
    r'$f_{ee}$',      # y label  
    r'$t \, [s]$',                          # t label
    f"plots/fermi-dirac_f_ee_time.pdf",
    xlog= False, 
    ylog= False, 
    ylowerlimit=-10.0, 
    yupperlimit=-1.3,
    xlowerlimit=None,
    xupperlimit=None
)

In [ ]:
def fermi_dirac_occupation(t, energy, mu, temperature, kappa_invcm, speed):
    """Time-dependent occupation for f(E,0)=0: f_eq(E) * [1 - exp(-c kappa(E) t)].

    energy, mu, and temperature are in MeV. kappa_invcm is the absorption
    inverse mean free path in 1/cm. speed is usually PhysConst.c (cm/s).
    A 1D time array returns shape (n_t, n_E).
    """
    t = np.asarray(t, dtype=float)
    energy = np.asarray(energy, dtype=float)
    kappa = np.asarray(kappa_invcm, dtype=float)
    f_eq = 1.0 / (1.0 + np.exp((energy - mu) / temperature))
    rate = speed * kappa
    if t.ndim == 0:
        return f_eq * (1.0 - np.exp(-rate * t))
    return f_eq * (1.0 - np.exp(-rate * t.reshape(-1, 1)))

# time_s = np.linspace(0.0, time_s[-1]/100, 10000)

# k_*_invcm lists NuLib groups 4-17; cut energies are groups 5-17.
f_ee_all_time_theory = fermi_dirac_occupation(
    time_s,
    energies_center_Mev,
    u_ee_MeV,
    Temperature_Mev,
    k_ee_invcm[1:],
    pc.PhysConst.c,
)

In [ ]:
plots_dir = Path("plots")
plots_dir.mkdir(exist_ok=True)

norm = plt.Normalize(energies_center_Mev.min(), energies_center_Mev.max())
cmap = plt.cm.viridis

def plot_fee_all_bins(f_num, f_theory, ylabel, filename):
    """One curve per energy bin: analytic and numerical occupation vs time."""
    fig, ax = plt.subplots(figsize=(12, 8))
    f_num = np.asarray(f_num, dtype=float)
    f_theory = np.asarray(f_theory, dtype=float)

    for ibin, E_MeV in enumerate(energies_center_Mev):
        color = cmap(norm(E_MeV))
        ax.plot(
            np.log10(time_s), f_theory[:, ibin]/f_eq_ee[ibin],
            color=color, linestyle="-", linewidth=4, alpha=0.4,
        )
        ax.plot(
            np.log10(time_s), f_num[:, ibin]/f_eq_ee[ibin],
            color=color, linestyle=":", linewidth=4, alpha=1.0,
        )

    fs = 30
    ax.set_xlabel(r"$\log_{10}\,t \, [\mathrm{s}]$", fontsize=fs)
    ax.set_ylabel(ylabel, fontsize=fs)
    ax.set_xlim(np.log10(time_s[1]), np.log10(time_s[-1]))
    # ax.set_xscale("log")
    ax.tick_params(axis="both", which="major", labelsize=fs)
    ax.tick_params(axis="both", which="minor", labelsize=fs)

    from matplotlib.lines import Line2D
    legend_handles = [
        Line2D([0], [0], color="black", linestyle="-", linewidth=3, alpha=0.4,
               label=r"Analytical solution"),
        Line2D([0], [0], color="black", linestyle=":", linewidth=3,
               label=r"Numerical solution"),
    ]
    leg = ax.legend(handles=legend_handles, framealpha=0.0, fontsize=25)
    pf.apply_custom_settings(ax, leg, log_scale_y=False)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax)
    cbar.set_label(r"$E\,[\mathrm{MeV}]$", fontsize=fs)
    cbar.ax.tick_params(labelsize=fs)

    fig.tight_layout()
    fig.savefig(plots_dir / filename, bbox_inches="tight")
    plt.show()
    plt.close(fig)

plot_fee_all_bins(
    f_ab_all_times[:, 0, 0, :],
    f_ee_all_time_theory,
    r"$f_{ee}/f_{ee}^{\mathrm{FD}}$",
    "fermi-dirac_f_ee_all_bins.pdf",
)

In [ ]:

f_ee_all_time_relative_error = np.abs( ( f_ee_all_time_theory - f_ab_all_times[:, 0, 0, :] ) / f_ee_all_time_theory )

fig, ax = plt.subplots(figsize=(12, 8))

for ibin, E_MeV in enumerate(energies_center_Mev):
    color = cmap(norm(E_MeV))
    ax.plot(np.log10(time_s), np.log10(f_ee_all_time_relative_error[:, ibin]), color=color, linestyle="-", linewidth=4, alpha=1.0)

fs = 30
# ax.set_xlim(0, 1)
# ax.set_xscale("log")
# ax.set_yscale("log")
ax.set_xlabel(r"$\log_{10}t \, [\mathrm{s}]$", fontsize=fs)
ax.set_ylabel(r"$\log_{10} \, \mathrm{Relative \, error}$", fontsize=fs)
ax.tick_params(axis="both", which="major", labelsize=fs)
ax.tick_params(axis="both", which="minor", labelsize=fs)

pf.apply_custom_settings(ax, None, log_scale_y=False)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label(r"$E\,[\mathrm{MeV}]$", fontsize=fs)
cbar.ax.tick_params(labelsize=fs)

fig.tight_layout()
fig.savefig(plots_dir / "fermi-dirac_f_ee_all_bins_relative_error.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)

